In [1]:
! pip install faiss-cpu mistralai

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/3 [invoke]
   ---------------------------------------- 0/3 [invoke]
   ---------------------------------------- 0/3 [invoke]
   ---------------------------------------- 0/3 [invoke]
   ---------------------------------------- 0/3 [invoke]
   -------------------------- ------------- 2/3 [mistralai]
   -------------------------- ------------- 2/3 [mistralai]
   -------------------------- ------------- 2/3 [mistralai]
   -------------------------- ------------- 2/3 [mistralai]
   -------------------------- ------------- 2/3 [mistralai]
   -------------------------- ------------- 2/3 [mistralai]
   -------------------------- ------------- 2/3 [mistralai]
   -------------------------- ------------- 2/3 [mistralai]
   -------------------------- ------------- 2/3 [mistralai]
   -------------------------- ------------- 2/3 [mistralai]
   -------------------------- ------

In [11]:
from mistralai import Mistral
import requests
import numpy as np
import faiss
import os
from getpass import getpass

api_key= getpass("guHtoqMpYipOLnfhIqpOTVTs9gAh0ezI")
client = Mistral(api_key=api_key)

In [12]:
response = requests.get('https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt')
text = response.text

In [13]:
f = open('essay.txt', 'w')
f.write(text)
f.close()

In [14]:
len(text)

75014

In [15]:
chunk_size = 2048
chunks = [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

In [16]:
len(chunks)

37

In [17]:
def get_text_embedding(input):
    embeddings_batch_response = client.embeddings.create(
          model="mistral-embed",
          inputs=input
      )
    return embeddings_batch_response.data[0].embedding

In [18]:
text_embeddings = np.array([get_text_embedding(chunk) for chunk in chunks])

In [19]:
text_embeddings.shape

(37, 1024)

In [20]:
text_embeddings

array([[-0.03979492,  0.07733154,  0.00013709, ..., -0.01274109,
        -0.02101135, -0.00264168],
       [-0.03152466,  0.07226562,  0.02961731, ..., -0.01079559,
        -0.01189423, -0.00821686],
       [-0.05905151,  0.06112671,  0.01206207, ..., -0.0226593 ,
         0.00488663, -0.00665283],
       ...,
       [-0.05477905,  0.06890869,  0.02703857, ..., -0.02456665,
        -0.02526855, -0.02687073],
       [-0.03884888,  0.05587769,  0.04718018, ..., -0.01812744,
         0.00926208, -0.00866699],
       [-0.03048706,  0.05831909,  0.01704407, ..., -0.01620483,
        -0.01800537, -0.04415894]], shape=(37, 1024))

In [21]:
d = text_embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(text_embeddings)

In [22]:
question = "What were the two main things the author worked on before college?"
question_embeddings = np.array([get_text_embedding(question)])
question_embeddings.shape

(1, 1024)

In [23]:
question_embeddings

array([[-0.05447388,  0.03479004,  0.0375061 , ..., -0.02787781,
        -0.00327492,  0.0029068 ]], shape=(1, 1024))

In [24]:
D, I = index.search(question_embeddings, k=2)
print(I)

[[0 3]]


In [25]:
retrieved_chunk = [chunks[i] for i in I.tolist()[0]]
print(retrieved_chunk)

['\n\nWhat I Worked On\n\nFebruary 2021\n\nBefore college the two main things I worked on, outside of school, were writing and programming. I didn\'t write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.\n\nThe first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district\'s 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain\'s lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.\n\nThe language we used was an early version of Fortran. You had to type programs on punch cards, then 

In [26]:
prompt = f"""
Context information is below.
---------------------
{retrieved_chunk}
---------------------
Given the context information and not prior knowledge, answer the query.
Query: {question}
Answer:
"""

In [27]:
def run_mistral(user_message, model="mistral-large-latest"):
    messages = [
        {
            "role": "user", "content": user_message
        }
    ]
    chat_response = client.chat.complete(
        model=model,
        messages=messages
    )
    return (chat_response.choices[0].message.content)

In [28]:
run_mistral(prompt)

'The two main things the author worked on before college were **writing** (specifically short stories) and **programming**.'